# Business Entity Resolution — Team Pipeline
**Amazon ML Challenge 2026**

This notebook is meant to be run inside a **SageMaker Studio Shared Space** so all 4 team members can co-edit it live. Suggested ownership (edit the cell, don't fork the notebook):

| Section | Owner | Stage |
|---|---|---|
| 1–2 | Person A | Setup, EDA, normalization |
| 3 | Person B | Blocking / candidate generation |
| 4–5 | Person C | Feature engineering + model training |
| 6–7 | Person D | Threshold tuning, validation scoring, final output + `validate_submission.py` |

Run cells top to bottom. `src/` modules hold the real logic — this notebook just orchestrates and visualizes, so keep experiments in scratch cells below each section rather than editing the `.py` files simultaneously (use git for that, see README).

## 1. Setup

In [1]:
# Run once per space / kernel restart
%pip install -q -r ../requirements.txt


Note: you may need to restart the kernel to use updated packages.


In [2]:
import sys, os
sys.path.insert(0, os.path.abspath("../src"))

import pandas as pd
import numpy as np
import importlib, blocking
importlib.reload(blocking)
import time

from io_utils import load_source, load_ground_truth, explode_ground_truth, write_pairs_tsv
from normalize_parallel import add_normalized_columns_parallel as add_normalized_columns   # <-- changed line
from blocking import build_candidates, candidates_to_frame, bucket_size_report
from features import build_feature_matrix, frame_to_lookup, FEATURE_NAMES
from train_model import build_training_table, train_lightgbm, predict_with_ensemble, feature_importance_report
from evaluate import macro_f05, sweep_threshold, blocking_recall

DATA_DIR = "/home/sagemaker-user/dataset/dataset"
TRAIN_DIR = os.path.join(DATA_DIR, "train")
TEST_DIR = os.path.join(DATA_DIR, "test")

## 2. Load data & EDA

In [3]:
train_s1 = load_source(os.path.join(TRAIN_DIR, "train_source1.tsv"))
train_s2 = load_source(os.path.join(TRAIN_DIR, "train_source2.tsv"))
train_s3 = load_source(os.path.join(TRAIN_DIR, "train_source3.tsv"))
gt = load_ground_truth(os.path.join(TRAIN_DIR, "train_ground_truth.tsv"))

print("S1:", train_s1.shape, "S2:", train_s2.shape, "S3:", train_s3.shape, "GT rows:", gt.shape)
print("Countries in train:", pd.concat([train_s1["country"], train_s2["country"], train_s3["country"]]).unique())
print("Singleton rate (S1 with no match):", (gt["matched_entity_ids"] == "").mean())


S1: (2206821, 4) S2: (5034616, 4) S3: (5285603, 4) GT rows: (2206821, 2)
Countries in train: ['US' 'India']
Singleton rate (S1 with no match): 0.05584820880352326


In [4]:
# Eyeball a few positive pairs to see the real noise patterns
gt_exploded = explode_ground_truth(gt)
sample = gt_exploded.sample(5, random_state=1)
for _, r in sample.iterrows():
    s1_row = train_s1[train_s1.entity_id == r.source1_entity_id]
    other_row = pd.concat([train_s2, train_s3])
    other_row = other_row[other_row.entity_id == r.matched_entity_id]
    print("S1 :", s1_row[["business_name","business_address","country"]].values)
    print("MAT:", other_row[["business_name","business_address","country"]].values)
    print("---")


S1 : [['Davis Society Care' '3424 Humboldt Avenue, Minneapolis, MN' 'US']]
MAT: [['Davis Society' 'MINNEAPOLIS, 424 HUMBOLDT AVE, MN' 'US']]
---
S1 : [['Peak Highland Fast Inc'
  '3029 Veterans Parkway, City Of Marshfield, WI' 'US']]
MAT: [['Peak Highland Fast Inc'
  '3029 Veterans Parkway, City Of Marshfield, Wisconsin' 'US']]
---
S1 : [['Coastal Maritime Inc' '805 Park Avenue, Unit A, Victoria, TX' 'US']]
MAT: [['Coastal Inc Services' 'Park Ave, Unit A, Vcitoria, Texas' 'US']]
---
S1 : [['Modern Foundation'
  'Delhi, New Delhi, 5, Ground Floor, Park End Colony, Vikas Marg, Preet Vihar, New Delhi'
  'India']]
MAT: [['मॉडर्न फाउंडेशन'
  'HN 372 5, Delhi, GROUND FLOOR, PARK END COLONY, VIKAS MARG, PREET VIHAR, NEW DELHI'
  'India']]
---
S1 : [['Internal Medicine Health LLC' '100 Sundown Drive, Unit #14, 06, IA'
  'US']]
MAT: [['Internal Medicine Health' '100 SUNDOWN DR, GLIDDEN, IA' 'US']]
---


## 3. Normalize + Blocking (candidate generation)\nThis stage sets your recall ceiling — check `blocking_recall` below before moving on.

In [5]:
import gc

print("Normalizing S1...")
train_s1n = add_normalized_columns(train_s1)
del train_s1; gc.collect()

print("Normalizing S2...")
train_s2n = add_normalized_columns(train_s2)
del train_s2; gc.collect()

print("Normalizing S3...")
train_s3n = add_normalized_columns(train_s3)
del train_s3; gc.collect()

train_other = pd.concat([train_s2n, train_s3n], ignore_index=True)
del train_s2n, train_s3n; gc.collect()

# Recall-safe sampling: keep every S2/S3 record referenced ANYWHERE in
# train_ground_truth.tsv (this guarantees sampling itself can never be
# the reason a true match is missing from blocking), then top up with
# random unmatched rows up to a memory-safe cap. This is different from
# both extremes we already ruled out:
#   - flat 3M random sample -> dropped true matches at random -> recall
#     capped ~0.25 (confirmed)
#   - full ~10-12M unsampled set -> OOM-crashed the kernel even on 500
#     S1 rows, because Signal 3/4 token-explode cost scales with the
#     size of train_other itself
matched_ids = set()
for ids in gt["matched_entity_ids"].dropna():
    if ids:
        matched_ids.update(ids.split(","))

is_matched = train_other["entity_id"].isin(matched_ids)
matched_other = train_other[is_matched]
unmatched_other = train_other[~is_matched]

SAFE_CAP = 5_000_000  # tune down to 3_000_000 if this still crashes
remaining_budget = max(0, SAFE_CAP - len(matched_other))
if len(unmatched_other) > remaining_budget:
    sampled_unmatched = unmatched_other.sample(n=remaining_budget, random_state=42)
else:
    sampled_unmatched = unmatched_other

train_other = pd.concat([matched_other, sampled_unmatched], ignore_index=True)
del matched_other, unmatched_other, sampled_unmatched
gc.collect()

print(f"train_other size: {len(train_other):,} rows")
print(f"All {len(matched_ids):,} true-match IDs retained (recall ceiling no longer capped by sampling)")

Normalizing S1...
Processing 2,206,821 records in 45 chunks using 16 workers...


Normalizing S2...
Processing 5,034,616 records in 101 chunks using 16 workers...
Normalizing S3...
Processing 5,285,603 records in 106 chunks using 16 workers...
train_other size: 7,638,365 rows
All 7,638,365 true-match IDs retained (recall ceiling no longer capped by sampling)


In [6]:
from sklearn.model_selection import train_test_split

s1_ids_all = train_s1n["entity_id"].values
s1_train_ids, s1_val_ids = train_test_split(s1_ids_all, test_size=0.2, random_state=42)

s1_train = train_s1n[train_s1n.entity_id.isin(s1_train_ids)].reset_index(drop=True)
s1_val = train_s1n[train_s1n.entity_id.isin(s1_val_ids)].reset_index(drop=True)
del train_s1n; gc.collect()

gt_train = gt[gt.source1_entity_id.isin(s1_train_ids)].reset_index(drop=True)
gt_val = gt[gt.source1_entity_id.isin(s1_val_ids)].reset_index(drop=True)
gt_exploded_train = explode_ground_truth(gt_train)

print(f"Train: {len(s1_train):,} entities, Val: {len(s1_val):,} entities")


Train: 1,765,456 entities, Val: 441,365 entities


### Sanity-check bucket sizes before full candidate build

Run this FIRST to verify blocking key selectivity. With prefix_len=6, top buckets should be <0.2% each.

In [7]:
bucket_size_report(train_other, prefix_len=6)


Total records: 7,638,365 | unique buckets: 554,956 | avg bucket size: 13.8
Top 15 largest buckets:
  'US|pediat': 46,450 records (0.61%)
  'US|family': 20,262 records (0.27%)
  'US|bright': 17,801 records (0.23%)
  'US|intern': 16,473 records (0.22%)
  'US|physic': 16,340 records (0.21%)
  'US|dental': 15,870 records (0.21%)
  'US|cardio': 15,789 records (0.21%)
  'US|dermat': 15,773 records (0.21%)
  'US|primar': 15,653 records (0.20%)
  'US|behavi': 15,308 records (0.20%)
  'US|vision': 15,244 records (0.20%)
  'US|chirop': 15,140 records (0.20%)
  'US|surgic': 15,017 records (0.20%)
  'US|womens': 14,975 records (0.20%)
  'India|mumbai': 14,947 records (0.20%)


US|pediat       46450
US|family       20262
US|bright       17801
US|intern       16473
US|physic       16340
                ...  
India|wgu in        1
US|zehphl           1
India|ktp ed        1
India|zw equ        1
US|688284           1
Name: count, Length: 554956, dtype: int64

In [8]:

# small, safe sanity check with country-gated + tightened blocking
test_candidates = build_candidates(s1_train.head(500), train_other, prefix_len=6, max_block_size=300)
print(f"Avg candidates per entity: {np.mean([len(v) for v in test_candidates.values()]):.1f}")
del test_candidates

Avg candidates per entity: 151.2


In [9]:
start = time.time()
c = build_candidates(s1_val, train_other, prefix_len=6, max_block_size=300, require_country_match=False)
elapsed = time.time() - start
r = blocking_recall(gt_val, c)
avg = np.mean([len(v) for v in c.values()])
print(f"recall={r:.4f}  avg_cands={avg:.1f}  time={elapsed:.1f}s")

recall=0.8025  avg_cands=156.8  time=305.2s


In [10]:
per_row_time = elapsed / len(s1_val)
full_size = len(s1_train)  # ~1.76M
print(f"Estimated full train build: {per_row_time * full_size / 60:.1f} minutes")

Estimated full train build: 20.3 minutes


In [11]:
import random

def diagnose_misses(gt_df, candidates, s1_df, other_df, n_samples=15):
    other_lookup = other_df.set_index("entity_id")[["business_name", "business_address", "country"]]
    s1_lookup = s1_df.set_index("entity_id")[["business_name", "business_address", "country"]]

    missed = []
    for _, row in gt_df.iterrows():
        s1_id = row["source1_entity_id"]
        true_ids = [x for x in str(row["matched_entity_ids"]).split(",") if x]
        if not true_ids:
            continue
        cand_ids = set(candidates.get(s1_id, []))
        for tid in true_ids:
            if tid not in cand_ids:
                missed.append((s1_id, tid))

    print(f"Total missed true-match pairs: {len(missed):,}")
    sample = random.sample(missed, min(n_samples, len(missed)))
    for s1_id, tid in sample:
        s1_row = s1_lookup.loc[s1_id]
        try:
            other_row = other_lookup.loc[tid]
        except KeyError:
            print(f"{tid} not found in other_df at all -- check ID scoping")
            continue
        print("-" * 80)
        print(f"S1  [{s1_id}] {s1_row['business_name']!r} | {s1_row['business_address']!r} | {s1_row['country']}")
        print(f"TRUE[{tid}] {other_row['business_name']!r} | {other_row['business_address']!r} | {other_row['country']}")

# 'c' is the candidates dict from your last completed test run — reuse it directly
diagnose_misses(gt_val, c, s1_val, train_other, n_samples=20)

Total missed true-match pairs: 301,640
--------------------------------------------------------------------------------
S1  [S1-757193440] 'South Allied Oak PC' | '973 Hicks Pike, Walton, KY' | US
TRUE[S2-611571517] 'South  Allied Oak' | 'KY, 973 HICKS PIKE, WALTON' | US
--------------------------------------------------------------------------------
S1  [S1-672378809] 'Lakshmi Software Pvt Ltd' | '30, Ram Kamal Sen Lane, Kolkata, Kolkata, Howrah, West Bengal' | India
TRUE[S3-897649582] 'lakshmisoftware.com' | 'WB, 0030, Kolkata, Howrah' | India
--------------------------------------------------------------------------------
S1  [S1-641051618] 'Western Institute of Technology, Inc' | '106 Honea Road, Hazel Green, AL' | US
TRUE[S3-491083982] 'Western Institute of Technology,' | '106 Honae Rd, N/A, Hazel GREN, Alabama' | US
--------------------------------------------------------------------------------
S1  [S1-917120296] 'Creative Automotive' | '2120 Lebanon Pike, Nashville, TN, Unit 50

In [ ]:
# OPTIMIZED: Use sample for development, full dataset for final run
SAMPLE_SIZE = None  # Set to None for full dataset

if SAMPLE_SIZE:
    s1_train_work = s1_train.sample(n=min(SAMPLE_SIZE, len(s1_train)), random_state=42)
    print(f"Development mode: using {len(s1_train_work):,} sample")
else:
    s1_train_work = s1_train
    print(f"Full mode: using all {len(s1_train):,} records")

print("Building candidates for training set...")
candidates_train = build_candidates(s1_train_work, train_other, prefix_len=6,
                                     max_block_size=300, require_country_match=False)

print("Building candidates for validation set...")
candidates_val = build_candidates(s1_val, train_other, prefix_len=6,
                                   max_block_size=300, require_country_match=False)

print("\nBlocking recall ceiling (train):", blocking_recall(gt_train, candidates_train))
print("Blocking recall ceiling (val):  ", blocking_recall(gt_val, candidates_val))
print(f"Avg candidates per S1 entity: {np.mean([len(v) for v in candidates_train.values()]):.1f}")

Full mode: using all 1,765,456 records
Building candidates for training set...


*If recall ceiling is well below ~0.95, loosen `sim_threshold` or raise `top_k` in `build_candidates` before tuning the model — no model can recover matches blocking never proposed.*

## 4. Feature engineering + training data

In [ ]:
s1_lookup_train = frame_to_lookup(s1_train)
other_lookup = frame_to_lookup(train_other)

X_train, y_train, groups_train, pairs_train_df = build_training_table(
    candidates_train, gt_exploded_train, s1_lookup_train, other_lookup
)
print("Training pairs:", X_train.shape, "| positive rate:", y_train.mean())


## 5. Train the matching model (LightGBM, group-aware CV)

In [ ]:
models, oof_pred = train_lightgbm(X_train, y_train, groups_train, n_splits=5)
print(feature_importance_report(models))


## 6. Threshold tuning on the held-out validation split (optimize macro F0.5 directly)

In [ ]:
s1_lookup_val = frame_to_lookup(s1_val)
val_pairs_df = candidates_to_frame(candidates_val, list(s1_val["entity_id"].values))
val_pairs_df = val_pairs_df.rename(columns={"candidate_entity_id": "candidate_entity_id"})

X_val = build_feature_matrix(val_pairs_df, s1_lookup_val, other_lookup)
val_pairs_df["prob"] = predict_with_ensemble(models, X_val)

cand_scores = {}
for s1_id, group in val_pairs_df.groupby("source1_entity_id"):
    cand_scores[s1_id] = list(zip(group["candidate_entity_id"], group["prob"]))

best_t, best_score, all_results = sweep_threshold(gt_val, list(s1_val["entity_id"].values), cand_scores)
print("Best threshold:", best_t, "-> validation macro F0.5:", best_score)
for t, s in all_results:
    print(f"  t={t:.2f}  F0.5={s:.4f}")


## 7. Generate final TEST outputs\nRe-blocks and re-scores on the real test set (which includes France) and writes both required files.

In [ ]:
from generate_outputs import run_test_pipeline

run_test_pipeline(
    test_dir=TEST_DIR,
    models=models,
    best_threshold=best_t,
    output_dir="../output",
    top_k=15,
    sim_threshold=0.35,
)


In [ ]:
# Validate locally before uploading to the leaderboard / packaging the zip
!python ../utils/validate_submission.py --matching ../output/matching_results.tsv \
    --candidate ../output/candidate_pairs.tsv --test-dir ../dataset/test


## Notes for the team
- Re-running Section 3 with different `top_k` / `sim_threshold` values is the highest-leverage experiment — log the resulting recall ceiling + avg candidates per entity each time.
- Section 5 hyperparameters (`num_leaves`, `learning_rate`, `n_estimators`) are worth a small grid search — edit `train_model.py`'s `default_params`, don't hardcode overrides in-notebook so everyone benefits.
- Keep the train/val split (`random_state=42`) fixed across everyone's experiments so F0.5 numbers are comparable.
- When you're happy with a result, commit `src/` + this notebook to git before generating the final test outputs, so the run is reproducible for the submission package.